# Treasury yield-curve inversion

This notebook asks how cross-asset outcomes differ after the ten-year minus two-year
Treasury spread is inverted at a month-end decision date. Inversion is an observable
market-rate state, not a deterministic recession forecast or causal intervention.

The hypothesis, zero boundary, lag policy, staleness rule, outcomes, and sensitivity
grid were fixed before live results. Committed cells retain no provider values or
outputs; live data and figures exist only in the executing process.

**Primary protocol.** The unit is the last common ETF trading session in a complete
month. The primary spread must be available at least one calendar day before that
close. Inverted is the treatment, noninverted is the reference, and the primary
horizon is one calendar month. The four two-sided primary outcomes are `SPY`, `TLT`,
`TLT-SHY`, and `TLT-IEF`; “two-sided” predeclares a difference without a directional
sign. Bonferroni intervals cover this family. Other assets, longer horizons,
thresholds, timing policies, and revision views are exploratory.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from studies._support import (
    CORE_SYMBOLS,
    STUDY_START,
    acquire_latest_series,
    acquire_monthly_prices,
    acquire_vintage_histories,
    assert_feature_panel_timing,
    build_point_in_time_levels,
    classification_transition_table,
    combined_outcome_labels,
    compare_statistics,
    configure_plots,
    familywise_primary_intervals,
    feature_provenance_summary,
    forward_labels,
    latest_revised_counterpart,
    momentum_baseline,
    open_live_session,
    plot_coverage,
    plot_feature_comparison,
    plot_normalized_prices,
    plot_regime_contrasts,
    plot_regime_distributions,
    plot_regime_means,
    plot_regime_timeline,
    plot_revision_gap,
    plot_sample_sizes,
    plot_sensitivity_heatmap,
    regime_contrast_statistics,
    regime_statistics,
    return_spread_labels,
    simultaneous_interval_family,
    study_run_manifest,
    temporal_contrast_stability,
    unconditional_statistics,
    validate_study_outputs,
)

configure_plots()
pd.set_option("display.max_columns", 20)
session = open_live_session()

## Hypothesis, universe, and maturity exposure

The fixed core universe is extended with `TLT` and `SHY` so long- and short-duration
Treasury outcomes can be compared with `IEF`. `SPY`, `GLD`, `DBC`, and `UUP` retain
broader cross-asset context. These ETFs are current products selected with hindsight;
their joint history is not a reconstructed investable universe.

`T10Y2Y` is the ten-year Treasury constant-maturity rate minus the two-year rate. The
hypothesis is that inversion is associated with different subsequent equity and
Treasury-duration outcomes. The study does not assume which month a recession begins
or that the spread alone has stable forecast value.

`TLT-SHY` and `TLT-IEF` subtract forward returns on the same dates. They preserve
common rate shocks and directly measure whether added maturity exposure changes the
inversion contrast. The ETF products have evolving duration and holdings, so these
are maturity-exposure proxies rather than constant-maturity bonds.

In [ ]:
symbols = (*CORE_SYMBOLS, "TLT", "SHY")
price_history, market_provenance = acquire_monthly_prices(session, symbols)
prices = price_history.loc[STUDY_START:]
histories = acquire_vintage_histories(
    session,
    ("T10Y2Y",),
    prices.index,
    selected_view_series=frozenset({"T10Y2Y"}),
)
latest = acquire_latest_series(session, ("T10Y2Y",))
staleness = {"T10Y2Y": pd.Timedelta(days=7)}
point_in_time = build_point_in_time_levels(
    histories,
    prices.index,
    staleness,
    latest_nonmissing_series=frozenset({"T10Y2Y"}),
)
latest_levels = latest_revised_counterpart(point_in_time, latest)
feature_provenance = feature_provenance_summary(point_in_time)
manifest = study_run_manifest(
    prices,
    series_ids=("T10Y2Y",),
    thresholds="primary inversion=spread below 0 percentage points",
    staleness=staleness,
)
display(manifest, market_provenance, feature_provenance)

## Selected historical views and coverage

The daily spread is requested through three bounded sets of historical views: the
actual common-close dates and their one- and two-day cutoffs. This covers every cutoff
used by the timing grid without requesting an unbounded history. Combined provenance
retains the exact cutoff sets and query retrieval times. Persistra applies the chosen
operational lag and selects the newest nonmissing observation no more than seven days
old.

The staleness allowance covers weekends and ordinary market holidays. A longer outage
produces a missing feature instead of carrying an obsolete rate forward.

FRED represents some daily holidays as explicit missing observations. For this daily
market series only, the study removes those rows and searches backward for the latest
valid spread; monthly macro studies do not use that fallback. Normalized market and
rate results also round-trip through temporary DuckDB storage before analysis.

Separate bounded requests can repeat one revision with a query-scoped interval end.
The combiner requires every economic field to agree, orders the distinct selected
revision starts, and assigns each inclusive interval through the day before the next
start; the last selected revision remains open. This reconstruction is sufficient for
the exact cutoff set queried here and is not presented as a complete daily archive.

In [ ]:
figure, _ = plot_coverage(market_provenance, feature_provenance)
plt.show()
plt.close(figure)

## Point-in-time rate information

Treasury market series are revised less extensively than many macro releases, but
point-in-time selection still matters: corrections, publication timing, weekends, and
retrieval-date history are distinct concepts. The latest-revised diagnostic preserves
the observation selected in real time and substitutes the current value for that same
date.

If the two paths overlap, that is an empirical property of this series and window. It
does not justify applying latest history to a revised labor, inflation, or GDP series.

In [ ]:
point_spread = point_in_time.frame.rename(columns={"T10Y2Y": "10y minus 2y spread"})
revised_spread = latest_levels.set_axis(point_spread.columns, axis="columns")
figure, _ = plot_feature_comparison(
    point_spread,
    revised_spread,
    ("10y minus 2y spread",),
)
plt.show()
plt.close(figure)

## Regime rule and timing convention

The primary regime is inverted when the admissible spread is below zero and
noninverted otherwise. Zero is a predeclared economic boundary, not a fitted cut point.
The price at decision month end and the lagged spread are both known before any
forward label begins. The timeline displays duration and clustering of inversion
episodes, which determine effective sample diversity.

A positive subsequent return in an inverted month does not mean inversion caused that
return, and a negative return does not invalidate the yield curve as a macro signal.

The primary state uses the strict rule \(s_d<0\), where \(s_d\) is the latest valid
spread known at the lagged cutoff. Zero belongs to the noninverted reference state.
The timeline makes uninterrupted inversion episodes visible; the later maturity-profile
plot then compares `SHY`, `IEF`, and `TLT` without treating their months as independent
macro events.

In [ ]:
def curve_regime(spread: pd.Series, *, boundary: float = 0.0) -> pd.Series:
    regime = pd.Series(pd.NA, index=spread.index, dtype="string")
    regime.loc[spread.notna() & spread.lt(boundary)] = "inverted"
    regime.loc[spread.notna() & spread.ge(boundary)] = "noninverted"
    return regime

def curve_threshold_state(spread: pd.Series, boundary: float) -> pd.Series:
    regime = pd.Series(pd.NA, index=spread.index, dtype="string")
    regime.loc[spread.notna() & spread.lt(boundary)] = "below boundary"
    regime.loc[spread.notna() & spread.ge(boundary)] = "at or above boundary"
    return regime

point_regimes = curve_regime(point_in_time.frame["T10Y2Y"])
latest_regimes = curve_regime(latest_levels["T10Y2Y"])
display(point_regimes.value_counts(dropna=False).rename("decision count"))
figure, _ = plot_regime_timeline(
    point_in_time.frame["T10Y2Y"],
    point_regimes,
    title="Point-in-time Treasury spread and inversion states",
    ylabel="Percentage points",
    boundaries=(0.0,),
)
plt.show()
plt.close(figure)

## Forward labels and time-series baselines

Persistra builds one-, three-, and twelve-month forward returns as label objects whose
ending dates are explicit. The macro feature cannot access them. Secondary horizons
overlap, while the primary one-month horizon does not overlap on monthly decisions.

The unconditional baseline establishes ordinary asset behavior over the same sample.
The twelve-month momentum baseline asks whether a conventional price-only state is at
least as descriptive as inversion. Normalized adjusted closes provide context without
constructing a strategy or charging transaction costs.

An \(h\)-month label is \(P_{d+h}/P_d-1\), and its stored ending close must fall
exactly \(h\) calendar months after the decision. Both baselines use dates with a valid
primary curve state. A one-year price warm-up supplies trailing momentum at the start
of the analysis window; normalized prices remain context rather than a backtest.

In [ ]:
labels = forward_labels(prices)
eligible = point_regimes.notna()
unconditional = unconditional_statistics(labels, eligible=eligible)
momentum = momentum_baseline(price_history, labels, eligible=eligible)
display(unconditional, momentum)
figure, _ = plot_normalized_prices(prices)
plt.show()
plt.close(figure)

## Association estimates and uncertainty

The table reports every asset and horizon rather than highlighting the largest
difference. Means are accompanied by horizon-aware
heteroskedasticity-and-autocorrelation-consistent (HAC) intervals, counts, coverage,
volatility, and positive shares. One-month drawdown is calculated within contiguous
regime episodes so separated inversions are not compounded into one artificial path.

These estimates share macro episodes and market shocks. Confidence intervals quantify
sampling uncertainty under an approximation; they do not solve structural instability
or the multiple comparisons across assets and horizons.

Treated-minus-reference contrasts use a full-calendar HAC score series. The bandwidth
is at least the overlap floor and may be longer under the fixed automatic rule. The
four primary one-month outcomes receive Bonferroni simultaneous intervals; descriptive
group means remain pointwise, and longer horizons are exploratory. A gray contrast has
fewer than twelve outcomes or two outcome-eligible episodes on a side. That minimum is
only a display rule, especially because two episodes do not make normal inference
dependable. First/second-half and leave-one-episode-out estimates expose this risk.

In [ ]:
point_statistics = regime_statistics(labels, point_regimes)
latest_statistics = regime_statistics(labels, latest_regimes)
directional_contrasts = regime_contrast_statistics(
    labels,
    point_regimes,
    treated="inverted",
    reference="noninverted",
    assets=("SPY", "TLT"),
)
duration_labels = return_spread_labels(
    labels,
    {
        "TLT minus SHY": ("TLT", "SHY"),
        "TLT minus IEF": ("TLT", "IEF"),
    },
)
duration_contrasts = regime_contrast_statistics(
    duration_labels,
    point_regimes,
    treated="inverted",
    reference="noninverted",
)
primary_contrasts = familywise_primary_intervals(
    pd.concat([directional_contrasts, duration_contrasts], ignore_index=True)
)
display(point_statistics, primary_contrasts)
figure, _ = plot_regime_means(
    point_statistics,
    title="Yield-curve state and one-month outcomes",
)
plt.show()
plt.close(figure)

figure, _ = plot_regime_contrasts(
    primary_contrasts,
    title="Predeclared inversion contrasts",
)
plt.show()
plt.close(figure)

duration_order = ("SHY", "IEF", "TLT")
duration_rows = point_statistics.loc[
    point_statistics["horizon_months"].eq(1)
    & point_statistics["asset"].isin(duration_order)
]
figure, axis = plt.subplots(figsize=(9, 5.2))
for regime, marker in (("inverted", "o"), ("noninverted", "s")):
    means = (
        duration_rows.loc[duration_rows["regime"].eq(regime)]
        .set_index("asset")
        .reindex(duration_order)["mean_return"]
    )
    axis.plot(
        duration_order,
        means,
        label=regime,
        marker=marker,
    )
axis.axhline(0, color="#333333", linewidth=0.9)
axis.set(
    title="Treasury maturity response by curve state",
    xlabel="Increasing maturity exposure",
    ylabel="One-month mean simple return",
)
axis.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axis.legend()
figure.tight_layout()
plt.show()
plt.close(figure)

## Outcome distributions and episode counts

Treasury duration is central to this question, so the distribution plot compares
equities with intermediate, long, and short Treasuries. Box plots show dispersion and
asymmetry that a mean can hide. The separate count chart makes the imbalance between
inverted and noninverted decisions visible.

No observation is removed because it looks exceptional. Crisis outcomes are part of
the phenomenon under study and must remain in both tables and figures.

In [ ]:
figure, _ = plot_regime_distributions(
    labels[1],
    point_regimes,
    assets=("SPY", "IEF", "TLT", "SHY"),
    title="One-month outcomes by yield-curve state",
)
plt.show()
plt.close(figure)

figure, _ = plot_sample_sizes(point_statistics)
plt.show()
plt.close(figure)

## Sensitivity to boundary and operational lag

The fixed grid crosses boundaries of -0.25, 0, and 0.25 percentage points with lags of
zero, one, and two days. Each cell is the `TLT` one-month mean below the boundary minus
its mean at or above the boundary. “Inverted” is reserved for the zero-boundary primary
rule. Rebuilding the feature panel for every lag ensures the
sensitivity test changes information availability rather than shifting a completed
feature after the fact.

The zero-day row is an explicitly optimistic, noncausal look-ahead diagnostic. FRED's
day-resolution availability cannot prove a same-day spread preceded the ETF close, so
that row is excluded from admissible timing conclusions. One day is the primary causal
policy and two days is conservative. Every panel undergoes an availability assertion.
The long table retains counts, episodes, HAC errors, and intervals; one exploratory
Bonferroni family covers all nine cells before the heatmap masks rows below the display
threshold. No cell becomes a replacement primary specification.

In [ ]:
sensitivity_rows = []
lag_labels = {
    0: "0 — optimistic same-day diagnostic",
    1: "1 — primary causal policy",
    2: "2 — conservative policy",
}
for lag_days in (0, 1, 2):
    panel = build_point_in_time_levels(
        histories,
        prices.index,
        {"T10Y2Y": pd.Timedelta(days=7)},
        publication_lag=pd.Timedelta(days=lag_days),
        latest_nonmissing_series=frozenset({"T10Y2Y"}),
    )
    assert_feature_panel_timing(panel)
    matched = panel.provenance["available_from"].notna()
    assert panel.provenance.loc[matched, "available_from"].le(
        panel.provenance.loc[matched, "decision_date"]
        - pd.Timedelta(days=lag_days)
    ).all()
    for boundary in (-0.25, 0.0, 0.25):
        regimes = curve_threshold_state(panel.frame["T10Y2Y"], boundary)
        table = regime_contrast_statistics(
            {1: labels[1]},
            regimes,
            treated="below boundary",
            reference="at or above boundary",
            assets=("TLT",),
        )
        table.insert(0, "boundary", boundary)
        table.insert(0, "lag_policy", lag_labels[lag_days])
        sensitivity_rows.append(table)
sensitivity_table = simultaneous_interval_family(
    pd.concat(sensitivity_rows, ignore_index=True),
    family_id="yield-curve lag and threshold family",
)
sensitivity = sensitivity_table.pivot(
    index="lag_policy", columns="boundary", values="mean_difference"
).where(
    sensitivity_table.pivot(
        index="lag_policy",
        columns="boundary",
        values="meets_display_threshold",
    )
)
display(sensitivity_table, sensitivity)
figure, _ = plot_sensitivity_heatmap(
    sensitivity,
    title="Long-Treasury contrast across timing policies",
    color_label="Difference in one-month mean return",
)
plt.show()
plt.close(figure)

## Latest-revised diagnostic

The comparison is retained even if this market-rate series has few revisions. It
records membership differences, conditional-mean changes, and the source-value gap.
The purpose is to demonstrate the same temporal discipline used for more heavily
revised series, including the possibility of a materially negative finding about
revision bias here.

The current series is never substituted into the primary regime classification.

The classification table includes an unclassified state, so missing-to-present
transitions are visible. The first/second-half and leave-one-episode-out table is a
real-time robustness diagnostic, not part of the revision comparison. A small revision
gap for this rate series would not weaken the need for vintage discipline elsewhere.

In [ ]:
revision_comparison = compare_statistics(point_statistics, latest_statistics)
classification_changes = classification_transition_table(
    point_regimes,
    latest_regimes,
)
stability_labels = combined_outcome_labels(
    labels,
    assets=("SPY", "TLT"),
    spreads={
        "TLT minus SHY": ("TLT", "SHY"),
        "TLT minus IEF": ("TLT", "IEF"),
    },
)
stability = temporal_contrast_stability(
    stability_labels,
    point_regimes,
    treated="inverted",
    reference="noninverted",
)
display(revision_comparison, classification_changes, stability)
figure, _ = plot_revision_gap(
    point_in_time.frame["T10Y2Y"],
    latest_levels["T10Y2Y"],
    title="Treasury-spread revision substitution gap",
)
plt.show()
plt.close(figure)

## Limitations and execution checks

The spread is one curve measure sampled monthly. It ignores the full yield curve,
term-premium estimates, inflation expectations, policy surprises, and intramonth
dynamics. ETF duration and composition change. If the live sample contains few
independent inversion episodes, month counts overstate event diversity. Outcomes
overlap at longer horizons, and many asset-horizon
comparisons increase false-discovery risk. Returns omit implementation costs.

The final audit checks positivity, alignment, provenance completeness, distinct label
objects, multiple regimes, and finite summaries. Availability assertions enforce the
one-day lag but cannot prove economic interpretation.

For background, FRED defines
[`T10Y2Y`](https://fred.stlouisfed.org/series/T10Y2Y) as the ten-year minus two-year
constant-maturity spread. The New York Fed's
[yield-curve guide](https://www.newyorkfed.org/research/capital_markets/ycfaq.htm)
illustrates why a term spread may be studied while also warning that a curve model is
not an official forecast. The exact maturity pair there differs from this notebook,
so it is conceptual context rather than validation.

In [ ]:
audit = validate_study_outputs(
    prices,
    point_in_time,
    labels,
    point_regimes,
    point_statistics,
    expected_regimes=frozenset({"inverted", "noninverted"}),
)
assert set(point_in_time.frame.columns).isdisjoint(labels[1].frame.columns)
matched = point_in_time.provenance["available_from"].notna()
assert point_in_time.provenance.loc[matched, "available_from"].le(
    point_in_time.provenance.loc[matched, "decision_date"] - pd.Timedelta(days=1)
).all()
display(audit)
session.close()

## Interpretation after execution

Inspect coverage and inversion episode counts first. Compare the entire distribution
with unconditional and price-momentum baselines, then examine uncertainty, the full
timing grid, and the latest-revised diagnostic. Preserve weak, unstable, or contrary
evidence. The notebook reports historical association; it does not certify recession
forecasts, market timing, causality, or strategy profitability.